## Hybrid Retriever- Combining Dense And Sparse Retriever

In [6]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from typing import List


In [7]:
# Step 1: Sample documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

# Step 2: Dense Retriever (FAISS + HuggingFace)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4290.40it/s]


In [10]:
### Sparse Retriever(BM25)
sparse_retriever=BM25Retriever.from_documents(docs)
sparse_retriever.k=3 ##top- k documents to retriever

## Step 4: Custom Hybrid Retriever (combining dense and sparse)
class HybridRetriever:
    def __init__(self, dense_retriever, sparse_retriever, dense_weight=0.7, sparse_weight=0.3):
        self.dense_retriever = dense_retriever
        self.sparse_retriever = sparse_retriever
        self.dense_weight = dense_weight
        self.sparse_weight = sparse_weight
    
    def invoke(self, query: str) -> List[Document]:
        dense_docs = self.dense_retriever.invoke(query)
        sparse_docs = self.sparse_retriever.invoke(query)
        
        # Combine and deduplicate
        seen = set()
        combined = []
        for doc in dense_docs:
            doc_id = doc.page_content
            if doc_id not in seen:
                combined.append(doc)
                seen.add(doc_id)
        for doc in sparse_docs:
            doc_id = doc.page_content
            if doc_id not in seen:
                combined.append(doc)
                seen.add(doc_id)
        return combined

hybrid_retriever = HybridRetriever(dense_retriever, sparse_retriever, dense_weight=0.7, sparse_weight=0.3)


In [11]:
hybrid_retriever

In [13]:
# Step 5: Query and get results
query = "How can I build an application using LLMs?"
results = hybrid_retriever.invoke(query)

# Step 6: Print results
for i, doc in enumerate(results):
    print(f"\n🔹 Document {i+1}:\n{doc.page_content}")


🔹 Document 1:
LangChain helps build LLM applications.

🔹 Document 2:
Langchain can be used to develop agentic ai application.

🔹 Document 3:
Pinecone is a vector database for semantic search.

🔹 Document 4:
Langchain has many types of retrievers.


### RAG Pipeline with hybrid retriever

In [18]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser


In [19]:
# Step 5: Prompt Template
prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

## step 6-llm
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
llm=ChatGroq(model="llama-3.3-70b-versatile",temperature=0.2)
llm


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000026E930E5940>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000026E931490A0>, model_name='llama-3.3-70b-versatile', temperature=0.2, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [20]:
### Create RAG Chain using LCEL
# Format documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain using LCEL
rag_chain = (
    RunnablePassthrough.assign(context=lambda x: format_docs(hybrid_retriever.invoke(x["input"])))
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain


RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(hybrid_retriever.invoke(x['input'])))
})
| PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\n\nContext:\n{context}\n\nQuestion: {input}\n')
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000026E930E5940>, async_client=<groq.resources.chat.co

In [21]:
# Step 9: Ask a question
query_input = {"input": "How can I build an app using LLMs?"}
answer = rag_chain.invoke(query_input)

# Step 10: Output
print("✅ Answer:\n", answer)

# Step 11: Show source documents
print("\n📄 Source Documents:")
source_docs = hybrid_retriever.invoke(query_input["input"])
for i, doc in enumerate(source_docs, 1):
    print(f"\nDoc {i}: {doc.page_content}")


✅ Answer:
 You can use LangChain to build an application using Large Language Models (LLMs). LangChain is a tool that helps develop LLM applications, and it also supports the development of agentic AI applications. Additionally, you can utilize LangChain's various retrievers, potentially in combination with a vector database like Pinecone for semantic search, to create a more robust and effective app.

📄 Source Documents:

Doc 1: LangChain helps build LLM applications.

Doc 2: Langchain can be used to develop agentic ai application.

Doc 3: Pinecone is a vector database for semantic search.

Doc 4: Langchain has many types of retrievers.
